# Case A End-to-End LLM Demo

This notebook is a local Mac / VS Code end-to-end demo for **Case A** of the ChatbotLP framework.

It shows the full pipeline from natural-language input to structured interpretation, validation, model construction, solve, primal and dual LaTeX output, theorem-style reasoning, economic interpretation, and a simple results figure.


In [ ]:
import os

os.environ["GEMINI_API_KEY"] = ""
os.environ["LLM_PROVIDER"] = "gemini"
os.environ["GEMINI_MODEL"] = "gemini-3-flash-preview"


## 1. Local environment setup

This section configures repo-local imports and checks whether the current Python environment can see a supported solver.


In [ ]:
from __future__ import annotations

import json
import platform
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import Markdown, display


def find_repo_root(path: Path) -> Path:
    for candidate in [path, *path.parents]:
        if (candidate / "src").is_dir():
            return candidate
    raise FileNotFoundError("Could not find the ChatbotLP repository root from the current working directory.")


REPO_ROOT = find_repo_root(Path.cwd().resolve())
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from src.chatbot_engine import run_chatbot_session
from src.figure_data import build_figure_data
from src.formal_context_builder import build_formal_math_context
from src.llm_problem_interpreter import interpret_problem_from_text, summarize_problem_state
from src.math_response_generator import MathResponseGenerator
from src.model_builder import build_model_from_market_instance
from src.response_generator import generate_response_with_metadata
from src.schema import ProblemState
from src.solver import _get_solver, solve_model
from src.solver_results import SolverResults
from src.theorem_checker import check_theorems
from src.validator import validate_state

solver_obj, solver_name, solver_path, solver_tried = _get_solver(
    solver_name="ipopt",
    fallback_solver="glpk",
    verbose=False,
)

print("Repository root:", REPO_ROOT)
print("Working directory:", Path.cwd())
print("Python executable:", sys.executable)
print("Python version:", sys.version.split()[0])
print("Platform:", platform.platform())
print("LLM provider:", os.getenv("LLM_PROVIDER"))
print("Gemini model:", os.getenv("GEMINI_MODEL"))
print("Gemini key present:", bool(os.getenv("GEMINI_API_KEY")))
print("Solver available:", solver_obj is not None)
print("Selected solver:", solver_name)
print("Solver executable:", solver_path)
print("Solver candidates tried:", solver_tried)

if solver_obj is None:
    raise RuntimeError(
        "No supported solver was found. Install GLPK locally on macOS, for example with `brew install glpk`, "
        "or configure another solver before running the end-to-end demo."
    )


def require_live_gemini() -> None:
    if os.getenv("LLM_PROVIDER", "").lower() != "gemini":
        raise RuntimeError("This demo expects LLM_PROVIDER='gemini'.")
    if not os.getenv("GEMINI_API_KEY"):
        raise RuntimeError("Paste a valid Gemini API key into the environment cell before running the LLM interpretation cells.")


def display_json(title: str, obj) -> None:
    print(title)
    print(json.dumps(obj, indent=2, default=str))


MATH_GENERATOR = MathResponseGenerator(use_llm=False)


## 2. Case A prose input

The prose below is intentionally simple and solver-ready: one product, one supplier node, one consumer node, and one transport link with no transformation technology.


In [ ]:
CASE_A_PROSE = """
Consider a coordinated supply chain with two nodes, N1 and N2, and one product P1.
At node N1 there is a supplier S1 that can supply up to 100 units of P1.
At node N2 there is a consumer C1 that can consume up to 60 units of P1.
There is a transport link T1 from N1 to N2 for product P1 with capacity 80 units.
Supplier S1 offers 100 units at a bid price of 10 per unit.
Consumer C1 is willing to pay 22 per unit for up to 60 units.
There are no transformation technologies in this benchmark case.
""".strip()

print(CASE_A_PROSE)


## 3. LLM interpretation: prose → semantic plan → `ProblemState`

This cell uses the Gemini-backed interpreter in [`src/llm_problem_interpreter.py`](/Users/mgomezochoa/Documents/ChatbotLP/src/llm_problem_interpreter.py) to produce a semantic plan and a structured `ProblemState`.


In [ ]:
require_live_gemini()

interpretation = interpret_problem_from_text(CASE_A_PROSE)
semantic_plan = interpretation["semantic_plan"]
problem_state = interpretation["problem_state"]
market_instance = interpretation["market_instance"]
state_summary = summarize_problem_state(problem_state)

print("Narrative interpretation:\n")
print(interpretation["narrative_interpretation"])
print("\nState summary:\n")
print(json.dumps(state_summary, indent=2, default=str))

display_json("\nSemantic plan:", semantic_plan)
display_json("\nProblemState as JSON:", problem_state.to_dict())


## 4. Validation and theorem checks

The framework treats the LLM as an interpreter, not the mathematical authority. This step validates the interpreted state and checks benchmark/theorem applicability deterministically.


In [ ]:
validation_result = validate_state(problem_state)
theorem_checks = check_theorems(problem_state)

display_json("Validation result:", validation_result)

theorem_rows = [
    {
        "theorem_id": check.theorem_id,
        "theorem_name": check.theorem_name,
        "applies": check.applies,
        "explanation": check.explanation,
    }
    for check in theorem_checks
]
display(pd.DataFrame(theorem_rows))

if not validation_result["solver_ready"]:
    raise RuntimeError("The interpreted Case A state is not solver-ready, so the end-to-end demo cannot continue.")


## 5. Model construction, solve, and classroom-style summary

This section reuses the model builder, solver, structured solver-results wrapper, and response generator.


In [ ]:
model = build_model_from_market_instance(market_instance)
solve_result = solve_model(model)
solver_results = SolverResults.from_solve_result(solve_result, problem_state)
figure_data = build_figure_data(solver_results)

context = {
    "type": "problem_formulation",
    "problem_state": problem_state,
    "market_instance": market_instance,
    "semantic_plan": semantic_plan,
    "narrative_interpretation": interpretation["narrative_interpretation"],
    "validation_result": validation_result,
    "state_summary": state_summary,
    "solve_result": solve_result.to_dict(),
    "solver_results": solver_results,
}

summary_text, summary_metadata = generate_response_with_metadata(
    mode="full",
    context=context,
    use_llm=False,
)

print("Solve success:", solve_result.success)
print("Solver status:", solve_result.status)
print("Objective value:", solve_result.objective_value)
display_json("\nRaw solve result:", solve_result.to_dict())
display(Markdown("## Classroom-style summary\n\n" + summary_text.replace("\n", "  \n")))


## 6. Structured solution tables

These tables expose the solver-backed quantities that the rest of the notebook will use.


In [ ]:
bid_acceptance_df = pd.DataFrame(figure_data["bid_acceptance_table"])
nodal_price_df = pd.DataFrame(figure_data["nodal_price_table"])
utilization_df = pd.DataFrame(figure_data["utilization_table"])
derived_metrics_df = pd.DataFrame(figure_data["derived_metrics_table"])

display(Markdown("### Bid acceptance"))
display(bid_acceptance_df)

display(Markdown("### Nodal price table"))
display(nodal_price_df)

display(Markdown("### Utilization"))
display(utilization_df)

display(Markdown("### Derived metrics"))
display(derived_metrics_df)


## 7. Primal and dual formulation in LaTeX

The next cells generate render-ready LaTeX from the structured current instance using the formal math context builder and deterministic math-response generator.


In [ ]:
primal_context = build_formal_math_context(
    state=problem_state,
    user_message="Write the primal formulation in LaTeX for this current problem.",
    pedagogical_mode="full",
)
dual_context = build_formal_math_context(
    state=problem_state,
    user_message="Write the dual formulation in LaTeX for this current problem.",
    pedagogical_mode="full",
)

primal_latex = MATH_GENERATOR.generate_primal_latex(primal_context)
dual_latex = MATH_GENERATOR.generate_dual_latex(dual_context)

display(Markdown("## Primal formulation\n\n" + primal_latex))
display(Markdown("## Dual formulation\n\n" + dual_latex))


## 8. Theorem / duality reasoning in the Sampat-style workflow

This is the theorem-oriented view of the same current instance: structured assumptions, current primal and dual scaffolds, and deterministic exposition for the supported Sampat-style scope.


In [ ]:
proof_context = build_formal_math_context(
    state=problem_state,
    user_message="Show Theorem 1 for this current problem and explain the strong duality reasoning in LaTeX.",
    pedagogical_mode="full",
)
proof_text = MATH_GENERATOR.generate_theorem_proof_latex(proof_context)

print("Assumptions verified:", proof_context.assumptions_verified)
print("Assumptions missing:", proof_context.assumptions_missing)
display(Markdown(proof_text))


## 9. Economic interpretation questions

These prompts treat the solved state as the current classroom problem and ask for economically meaningful explanations.


In [ ]:
economic_queries = [
    "Explain the economic meaning of the dual variables and node-product prices in this current problem.",
    "Why do the accepted quantities and transport flow make sense economically in this Case A benchmark?",
]

for query in economic_queries:
    result = run_chatbot_session(
        state=problem_state,
        user_message=query,
        mode="guided",
        use_llm=False,
    )
    display(Markdown(f"## Question\n\n{query}\n\n## Response\n\n{result['response']}"))


## 10. A simple figure of results

This figure uses the plotting-ready output layer so the notebook can double as a presentation artifact.


In [ ]:
plot_bid_df = bid_acceptance_df.copy()
plot_price_df = nodal_price_df.copy()

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].bar(plot_bid_df["bid_id"], plot_bid_df["accepted_quantity"], color=["#4C78A8", "#F58518"])
axes[0].set_title("Accepted bid quantities")
axes[0].set_xlabel("Bid")
axes[0].set_ylabel("Quantity")

price_labels = plot_price_df["node"] + ":" + plot_price_df["product"]
axes[1].bar(price_labels, plot_price_df["normalized_price"], color="#54A24B")
axes[1].set_title("Normalized node-product prices")
axes[1].set_xlabel("Node-product")
axes[1].set_ylabel("Price")

fig.suptitle("Case A end-to-end demo outputs")
plt.tight_layout()
plt.show()


## 11. Optional one-call orchestration check

The step-by-step cells above expose the full pipeline transparently. The cell below shows the same Case A prose through the higher-level chatbot engine in one call.


In [ ]:
require_live_gemini()

session_result = run_chatbot_session(
    state=ProblemState(),
    user_message=CASE_A_PROSE,
    mode="full",
    use_llm=True,
)

print("Intent:", session_result.get("intent"))
print("Success:", session_result.get("success"))
display_json("Response metadata:", session_result.get("response_metadata", {}))
display(Markdown("## Orchestrated response\n\n" + session_result.get("response", "")))
